# Qwen Portable Slice Row-Processing Run (Colab GPU)

This notebook runs one localized portable-slice Colab row-processing lane for Task 103.

Invariants:
- Hemma remains the only place that selects rows.
- This notebook consumes one Hemma-issued portable slice bundle.
- The notebook is only an orchestrator around repo-owned script surfaces.
- Output must match the canonical Task 103 row-processing run-root contract.


## Hemma preparation (run in the Hemma repo clone before opening this notebook)

Prepare one fresh bounded `rixvox train` source-selection universe, assign Colab `slice_index=1`, then package the portable slice bundle into the repo for Colab:

```bash
pdm run task-103-preprocess-public-corpus launch \
  --task103-stage source-selection \
  --launch-id task129-colab-scale-selection-launch-20260311a \
  --task103-run-id task129-colab-scale-selection-20260311a \
  --rixvox-split train \
  --rixvox-max-rows-per-split 36000 \
  --skip-build

pdm run task-114-isolated-stages status \
  --launch-root build/verification/task-114-qwen-isolated-stages/task129-colab-scale-selection-launch-20260311a

pdm run task-121-colab-slice-bundle plan \
  --source-run-root /srv/scratch/sir-convert-a-lot/build/runs/qwen3-tts-swedish-preprocessing/task129-colab-scale-selection-20260311a \
  --output-root /srv/scratch/sir-convert-a-lot/build/reference/qwen3-tts-colab-slices/task129-scale-slice-1-of-2-20260311a \
  --slice-count 2 \
  --slice-index 1

mkdir -p colab_ml_training/proof_inputs
tar -C /srv/scratch/sir-convert-a-lot/build/reference/qwen3-tts-colab-slices/task129-scale-slice-1-of-2-20260311a \
  -czf colab_ml_training/proof_inputs/task129-scale-slice-1-of-2-20260311a-bundle.tar.gz \
  .
```

Wait for the source-selection launch to report `status=completed` before planning the slice. For sessions 2 and 3, keep the same `SLICE_ID`, bundle filename, persistent root, and run id so `--resume-row-processing` continues the same disjoint Colab-owned slice.

The bundle must unpack these three files under `SLICE_ROOT` in Colab:
- `selected_source_records.jsonl`
- `required_hub_files.json`
- `slice_summary.json`


In [1]:
%pip install -q accelerate datasets huggingface_hub jiwer librosa \
    onnxruntime pyarrow python-dotenv qwen-tts safetensors \
    sentencepiece soundfile sox transformers

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None or importlib.util.find_spec("torchaudio") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch", "torchaudio"],
        check=True,
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.1 MB/s eta 0:00:00


In [2]:
import json
import os
import subprocess
import sys
import tarfile
import time
from pathlib import Path

REPO_CLONE_URL = os.environ.get(
    "SIR_CONVERT_A_LOT_REPO_URL",
    "https://github.com/paunchygent/sir-convert-a-lot.git",
)


def _candidate_repo_roots() -> list[Path]:
    candidates: list[Path] = []
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)
    candidates.append(Path("/content/sir-convert-a-lot"))
    return candidates


def _refresh_repo_checkout(repo_root: Path) -> None:
    subprocess.run(["git", "fetch", "origin", "main"], check=True, cwd=repo_root)
    subprocess.run(["git", "checkout", "main"], check=True, cwd=repo_root)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True, cwd=repo_root)


def _resolve_repo_root() -> Path:
    for candidate in _candidate_repo_roots():
        if (candidate / "scripts" / "sir_convert_a_lot").exists():
            _refresh_repo_checkout(candidate)
            return candidate
    target = Path("/content/sir-convert-a-lot")
    if not target.exists():
        subprocess.run(
            [
                "git",
                "clone",
                REPO_CLONE_URL,
                target.as_posix(),
            ],
            check=True,
        )
    _refresh_repo_checkout(target)
    return target


REPO_ROOT = _resolve_repo_root()
os.chdir(REPO_ROOT)
SLICE_ID = os.environ.get(
    "SIR_COLAB_SLICE_ID",
    "task138-task129-remaining-unique-20260311a",
)
BUNDLE_FILENAME = os.environ.get(
    "SIR_COLAB_BUNDLE_FILENAME",
    "task138-task129-remaining-unique-20260311a-bundle.tar.gz",
)
RUN_ID = os.environ.get(
    "SIR_COLAB_RUN_ID",
    "task129-colab-scale-rowproc-1-of-2-20260311a",
)
PERSIST_ROOT = Path(
    os.environ.get(
        "SIR_COLAB_PERSIST_ROOT",
        "/content/drive/MyDrive/sir-convert-a-lot",
    )
)
if (
    PERSIST_ROOT.as_posix().startswith("/content/drive")
    and not Path("/content/drive/MyDrive").exists()
):
    try:
        from google.colab import drive
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "Google Drive is required for the multi-session Colab slice, "
            "but google.colab is unavailable in this runtime."
        ) from exc
    drive.mount("/content/drive")
if (
    PERSIST_ROOT.as_posix().startswith("/content/drive")
    and not Path("/content/drive/MyDrive").exists()
):
    raise RuntimeError(
        "Google Drive mount did not expose /content/drive/MyDrive, "
        "so the persistent RUN_ROOT is unavailable."
    )
SLICE_ROOT = REPO_ROOT / "colab_inputs" / SLICE_ID
PROOF_BUNDLE_PATH = REPO_ROOT / "colab_ml_training" / "proof_inputs" / BUNDLE_FILENAME
DATA_ROOT = Path("/content/data/qwen3-tts-swedish-corpus")
RUN_ROOT = PERSIST_ROOT / "work/runs" / RUN_ID
OUTPUT_ROOT = PERSIST_ROOT / "work/reference/qwen3-tts-swedish-corpus"
CACHE_DIR = Path("/content/cache/huggingface")
ROWPROC_STDOUT_PATH = RUN_ROOT / "row_processing.stdout.log"
ROWPROC_STDERR_PATH = RUN_ROOT / "row_processing.stderr.log"
LOCALIZED_SELECTED_SOURCE_RECORDS_PATH = SLICE_ROOT / "localized_selected_source_records.jsonl"
PLANNED_SOURCE_SELECTION_CAP = 36_000
PLANNED_SLICE_ROW_TARGET = 7_187
ROWPROC_TIMEOUT_SECONDS = 11 * 60 * 60
ROW_WORKER_COUNT = 10
GPU_ASR_WORKER_COUNT = 2

for path in (SLICE_ROOT, DATA_ROOT, RUN_ROOT.parent, OUTPUT_ROOT.parent, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(
    json.dumps(
        {
            "repo_root": REPO_ROOT.as_posix(),
            "repo_clone_url": REPO_CLONE_URL,
            "slice_id": SLICE_ID,
            "bundle_filename": BUNDLE_FILENAME,
            "run_id": RUN_ID,
            "slice_root": SLICE_ROOT.as_posix(),
            "proof_bundle_path": PROOF_BUNDLE_PATH.as_posix(),
            "persist_root": PERSIST_ROOT.as_posix(),
            "data_root": DATA_ROOT.as_posix(),
            "run_root": RUN_ROOT.as_posix(),
            "output_root": OUTPUT_ROOT.as_posix(),
            "cache_dir": CACHE_DIR.as_posix(),
            "rowproc_stdout_path": ROWPROC_STDOUT_PATH.as_posix(),
            "rowproc_stderr_path": ROWPROC_STDERR_PATH.as_posix(),
            "localized_selected_source_records_path": (
                LOCALIZED_SELECTED_SOURCE_RECORDS_PATH.as_posix()
            ),
            "planned_source_selection_cap": PLANNED_SOURCE_SELECTION_CAP,
            "planned_slice_row_target": PLANNED_SLICE_ROW_TARGET,
            "rowproc_timeout_seconds": ROWPROC_TIMEOUT_SECONDS,
            "row_worker_count": ROW_WORKER_COUNT,
            "gpu_asr_worker_count": GPU_ASR_WORKER_COUNT,
        },
        indent=2,
    )
)

Mounted at /content/drive
{
  "repo_root": "/content/sir-convert-a-lot",
  "repo_clone_url": "https://github.com/paunchygent/sir-convert-a-lot.git",
  "slice_id": "task138-task129-remaining-unique-20260311a",
  "bundle_filename": "task138-task129-remaining-unique-20260311a-bundle.tar.gz",
  "run_id": "task129-colab-scale-rowproc-1-of-2-20260311a",
  "slice_root": "/content/sir-convert-a-lot/colab_inputs/task138-task129-remaining-unique-20260311a",
  "proof_bundle_path": "/content/sir-convert-a-lot/colab_ml_training/proof_inputs/task138-task129-remaining-unique-20260311a-bundle.tar.gz",
  "persist_root": "/content/drive/MyDrive/sir-convert-a-lot",
  "data_root": "/content/data/qwen3-tts-swedish-corpus",
  "run_root": "/content/drive/MyDrive/sir-convert-a-lot/work/runs/task129-colab-scale-rowproc-1-of-2-20260311a",
  "output_root": "/content/drive/MyDrive/sir-convert-a-lot/work/reference/qwen3-tts-swedish-corpus",
  "cache_dir": "/content/cache/huggingface",
  "rowproc_stdout_path": "/co

In [3]:
if not PROOF_BUNDLE_PATH.exists():
    raise FileNotFoundError("Expected committed proof bundle at " + PROOF_BUNDLE_PATH.as_posix())


def _extract_portable_slice_bundle(bundle_path: Path, slice_root: Path) -> None:
    slice_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(bundle_path, "r:gz") as archive:
        extract_kwargs: dict[str, str] = {}
        if sys.version_info >= (3, 12):
            extract_kwargs["filter"] = "data"
        archive.extractall(slice_root, **extract_kwargs)


_extract_portable_slice_bundle(PROOF_BUNDLE_PATH, SLICE_ROOT)

required_bundle_files = [
    SLICE_ROOT / "selected_source_records.jsonl",
    SLICE_ROOT / "required_hub_files.json",
]
missing_bundle_files = [path.as_posix() for path in required_bundle_files if not path.exists()]
if missing_bundle_files:
    raise FileNotFoundError(
        "Portable slice bundle extraction failed: " + ", ".join(missing_bundle_files)
    )

summary_candidates = [
    SLICE_ROOT / "slice_summary.json",
    SLICE_ROOT / "selected_source_records_dedupe_summary.json",
]
summary_path = next((path for path in summary_candidates if path.exists()), None)
if summary_path is None:
    raise FileNotFoundError("Portable slice bundle extraction failed: missing summary artifact")

slice_summary = json.loads(summary_path.read_text(encoding="utf-8"))
slice_summary

{'excluded_completed_row_count': 17994,
 'excluded_reserved_row_count': 0,
 'input_row_count': 18000,
 'input_selected_source_records_path': '/srv/scratch/sir-convert-a-lot/build/reference/qwen3-tts-colab-slices/task129-scale-slice-1-of-2-20260311a/selected_source_records.jsonl',
 'output_row_count': 7187,
 'output_selected_source_records_path': '/srv/scratch/sir-convert-a-lot/build/reference/qwen3-tts-colab-slices/task138-task129-remaining-unique-20260311a/selected_source_records.jsonl',
 'total_excluded_key_count': 15836}

## Stage required raw files and localize the slice

This uses only repo-owned portable-slice surfaces: first stage the exact Hub files listed in `required_hub_files.json`, then localize the selected slice into plain local audio files plus a persisted localized selected-source manifest.


In [4]:
stage_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle",
    "stage-required-files",
    "--slice-root",
    str(SLICE_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--cache-dir",
    str(CACHE_DIR),
]
localize_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle",
    "localize-slice",
    "--slice-root",
    str(SLICE_ROOT),
    "--data-root",
    str(DATA_ROOT),
]
print(" ".join(stage_command))
subprocess.run(stage_command, check=True, cwd=REPO_ROOT)
print(" ".join(localize_command))
subprocess.run(localize_command, check=True, cwd=REPO_ROOT)
assert LOCALIZED_SELECTED_SOURCE_RECORDS_PATH.exists(), (
    "Expected localized selected-source manifest at "
    + LOCALIZED_SELECTED_SOURCE_RECORDS_PATH.as_posix()
)

/usr/bin/python3 -m scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle stage-required-files --slice-root /content/sir-convert-a-lot/colab_inputs/task138-task129-remaining-unique-20260311a --data-root /content/data/qwen3-tts-swedish-corpus --cache-dir /content/cache/huggingface
/usr/bin/python3 -m scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle localize-slice --slice-root /content/sir-convert-a-lot/colab_inputs/task138-task129-remaining-unique-20260311a --data-root /content/data/qwen3-tts-swedish-corpus


## Run canonical Task 103 row-processing on the portable slice

This is the actual proof step. It must emit the same Task 103 run-root structure as Hemma row-processing while using the Colab GPU with one bounded but nontrivial ASR concurrency lane.


In [5]:
def _read_status_payload() -> dict[str, object] | None:
    status_path = RUN_ROOT / "status.json"
    if not status_path.exists():
        return None
    return json.loads(status_path.read_text(encoding="utf-8"))


def _spool_count() -> int:
    return sum(1 for _ in RUN_ROOT.rglob("spool/rows/**/*.json"))


def _audio_count() -> int:
    return sum(1 for _ in RUN_ROOT.rglob("audio_24k/**/*.wav"))


def _tail(path: Path, line_count: int = 80) -> str:
    if not path.exists():
        return "<missing>"
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-line_count:])


def _require_cuda_runtime() -> None:
    import torch

    try:
        nvidia_smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError(
            "Colab GPU preflight failed: `nvidia-smi` is unavailable. "
            "Switch the runtime to GPU and rerun from the bootstrap cell."
        ) from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "Colab GPU preflight failed: `nvidia-smi` returned a non-zero exit code. "
            "Switch the runtime to GPU and rerun from the bootstrap cell.\n"
            f"nvidia-smi stderr:\n{exc.stderr.strip()}"
        ) from exc

    cuda_available = bool(torch.cuda.is_available())
    device_count = int(torch.cuda.device_count())
    if not cuda_available or device_count <= 0:
        raise RuntimeError(
            "Colab GPU preflight failed: CUDA is not available to PyTorch. "
            "Switch the runtime to GPU and rerun from the bootstrap cell.\n"
            f"nvidia-smi:\n{nvidia_smi.stdout.strip()}\n"
            f"torch.cuda.is_available()={cuda_available}\n"
            f"torch.cuda.device_count()={device_count}"
        )

    print(
        json.dumps(
            {
                "cuda_available": cuda_available,
                "device_count": device_count,
                "device_name": torch.cuda.get_device_name(0),
                "nvidia_smi": nvidia_smi.stdout.strip(),
            },
            indent=2,
        )
    )


_require_cuda_runtime()


rowproc_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.run_task103_qwen_swedish_preprocessing",
    "--source-mode",
    "selected-source-records",
    "--selected-source-records-path",
    str(LOCALIZED_SELECTED_SOURCE_RECORDS_PATH),
    "--data-root",
    str(DATA_ROOT),
    "--run-root",
    str(RUN_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--stage",
    "row-processing",
    "--row-worker-count",
    str(ROW_WORKER_COUNT),
    "--gpu-asr-worker-count",
    str(GPU_ASR_WORKER_COUNT),
    "--resume-row-processing",
]
print(" ".join(rowproc_command))
RUN_ROOT.mkdir(parents=True, exist_ok=True)
with (
    ROWPROC_STDOUT_PATH.open("w", encoding="utf-8") as stdout_handle,
    ROWPROC_STDERR_PATH.open("w", encoding="utf-8") as stderr_handle,
):
    process = subprocess.Popen(
        rowproc_command,
        cwd=REPO_ROOT,
        stdout=stdout_handle,
        stderr=stderr_handle,
        text=True,
    )
    started_at = time.time()
    while True:
        returncode = process.poll()
        status_payload = _read_status_payload()
        print(
            json.dumps(
                {
                    "elapsed_seconds": round(time.time() - started_at, 1),
                    "returncode": returncode,
                    "status": (None if status_payload is None else status_payload.get("status")),
                    "processed_row_count": (
                        None
                        if status_payload is None
                        else status_payload.get("processed_row_count")
                    ),
                    "total_row_count": (
                        None if status_payload is None else status_payload.get("total_row_count")
                    ),
                    "current_dataset_row_id": (
                        None
                        if status_payload is None
                        else status_payload.get("current_dataset_row_id")
                    ),
                    "spool_rows": _spool_count(),
                    "audio_24k_files": _audio_count(),
                },
                indent=2,
            )
        )
        if returncode is not None:
            if returncode != 0:
                stdout_tail = _tail(ROWPROC_STDOUT_PATH)
                stderr_tail = _tail(ROWPROC_STDERR_PATH)
                raise RuntimeError(
                    "Row-processing failed.\nSTDOUT tail:\n"
                    + stdout_tail
                    + "\n\nSTDERR tail:\n"
                    + stderr_tail
                )
            break
        if time.time() - started_at > ROWPROC_TIMEOUT_SECONDS:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            stdout_tail = _tail(ROWPROC_STDOUT_PATH)
            stderr_tail = _tail(ROWPROC_STDERR_PATH)
            raise TimeoutError(
                "Row-processing timed out.\nSTDOUT tail:\n"
                + stdout_tail
                + "\n\nSTDERR tail:\n"
                + stderr_tail
            )
        time.sleep(20)
print({"elapsed_seconds": round(time.time() - started_at, 2)})

{
  "cuda_available": true,
  "device_count": 1,
  "device_name": "Tesla T4",
  "nvidia_smi": "Tesla T4, 15360 MiB"
}
/usr/bin/python3 -m scripts.sir_convert_a_lot.devops.run_task103_qwen_swedish_preprocessing --source-mode selected-source-records --selected-source-records-path /content/sir-convert-a-lot/colab_inputs/task138-task129-remaining-unique-20260311a/localized_selected_source_records.jsonl --data-root /content/data/qwen3-tts-swedish-corpus --run-root /content/drive/MyDrive/sir-convert-a-lot/work/runs/task129-colab-scale-rowproc-1-of-2-20260311a --output-root /content/drive/MyDrive/sir-convert-a-lot/work/reference/qwen3-tts-swedish-corpus --stage row-processing --row-worker-count 10 --gpu-asr-worker-count 2 --resume-row-processing
{
  "elapsed_seconds": 0.4,
  "returncode": null,
  "status": "running",
  "processed_row_count": 14613,
  "total_row_count": 7187,
  "current_dataset_row_id": "GX01MJU1-204-11",
  "spool_rows": 14613,
  "audio_24k_files": 14633
}
{
  "elapsed_seconds

In [6]:
status_payload = json.loads((RUN_ROOT / "status.json").read_text(encoding="utf-8"))
run_payload = json.loads((RUN_ROOT / "run.json").read_text(encoding="utf-8"))
spool_count = sum(1 for _ in RUN_ROOT.rglob("spool/rows/**/*.json"))
audio_count = sum(1 for _ in RUN_ROOT.rglob("audio_24k/**/*.wav"))
print(
    json.dumps(
        {
            "status": status_payload,
            "run": run_payload,
            "spool_rows": spool_count,
            "audio_24k_files": audio_count,
        },
        indent=2,
    )
)

{
  "status": {
    "completed_chunk_count": null,
    "completed_families": null,
    "current_chunk_index": null,
    "current_dataset_row_id": "H601AU1-158-9",
    "current_family": null,
    "current_parquet_batch_index": null,
    "current_split": null,
    "error": null,
    "processed_row_count": 15157,
    "promoted_root": "/content/drive/MyDrive/sir-convert-a-lot/work/reference/qwen3-tts-swedish-corpus",
    "required_audio_locator_count": null,
    "resolved_audio_locator_count": null,
    "run_id": "task129-colab-scale-rowproc-1-of-2-20260311a",
    "run_root": "/content/drive/MyDrive/sir-convert-a-lot/work/runs/task129-colab-scale-rowproc-1-of-2-20260311a",
    "selected_row_count": null,
    "source_mode": "selected-source-records",
    "stage": "row-processing",
    "status": "completed",
    "target_row_cap": null,
    "total_chunk_count": null,
    "total_row_count": 7187,
    "updated_at": "2026-03-12T10:39:17Z"
  },
  "run": {
    "generated_at": "2026-03-12T09:58:26Z

## Success criteria

A successful first Colab proof should show:
- valid `run.json`
- valid `status.json`
- non-empty `inventory/`
- non-empty `audio_24k/`
- non-empty `spool/rows/`
- no notebook-only preprocessing logic outside these repo-owned commands
